In [7]:
import os
import struct
import pandas as pd
from datetime import datetime

def read_tdx_day_file(file_path):
    """
    通用读取通达信 .day 文件（支持所有股票/ETF/新ETF）
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件 {file_path} 不存在")
    with open(file_path, 'rb') as f:
        buffer = f.read()

    num_bars = len(buffer) // 32
    data = []

    for i in range(num_bars):
        bar = buffer[i*32 : (i+1)*32]
        # 通达信 .day 固定结构
        date, open_, high, low, close, amount, vol, unused = struct.unpack("IIIIIfII", bar)

        # 日期转换
        dt = datetime(
            year=date//10000,
            month=(date%10000)//100,
            day=date%100
        )

        # 价格处理（关键！ETF 必须 /1000，股票 /100）
        if "sh56" in file_path or "sh58" in file_path or "sz159" in file_path:
            open_  /= 1000
            high   /= 1000
            low    /= 1000
            close  /= 1000
        else:
            open_  /= 100
            high   /= 100
            low    /= 100
            close  /= 100

        data.append([dt, open_, high, low, close, vol, amount])

    df = pd.DataFrame(data, columns=["datetime", "open", "high", "low", "close", "volume", "amount"])
    df.astype({
                    'open': 'float32',
                    'high': 'float32',
                    'low': 'float32',
                    'close': 'float32',
                    'volume': 'int64',  # 使用 int64 避免溢出
                    'amount': 'float32'
                })
    return df

In [8]:
df = read_tdx_day_file(r"D:\Users\lyx\Desktop\hsjday\vipdoc\sh\lday\sh563900.day")
df

,datetime,open,high,low,close,volume,amount
0,2025-05-07,1.007,1.010,1.002,1.007,202719600,203753056.0
1,2025-05-08,1.005,1.011,1.001,1.006,68869556,69481040.0
2,2025-05-09,1.008,1.012,1.008,1.012,53965656,54500984.0
3,2025-05-12,1.014,1.018,1.012,1.017,35756956,36276540.0
4,2025-05-13,1.018,1.027,1.018,1.022,30426500,31046510.0
...,...,...,...,...,...,...,...
218,2026-03-30,1.225,1.230,1.218,1.229,16781200,20552888.0
219,2026-03-31,1.234,1.241,1.221,1.224,39652200,48709272.0
220,2026-04-01,1.229,1.234,1.225,1.227,27422000,33705008.0
221,2026-04-02,1.227,1.233,1.222,1.228,36996200,45437336.0
